Opening few files to profile the data and inspect the schemas.

In [1]:
import polars as pl
import os
from dotenv import load_dotenv

In [4]:
import s3fs
load_dotenv()

import s3fs

fs = s3fs.S3FileSystem(
    key=os.getenv("AWS_ACCESS_KEY_ID"),
    secret=os.getenv("AWS_SECRET_ACCESS_KEY")
)

files = fs.glob("s3://techcatalyst-de-2026/raw/**/*.parquet")

for f in files:
    print(f)

print(f"Found {len(files)} parquet files")



techcatalyst-de-2026/raw/green_taxi/green_tripdata_2025-01.parquet
techcatalyst-de-2026/raw/green_taxi/green_tripdata_2025-02.parquet
techcatalyst-de-2026/raw/green_taxi/green_tripdata_2025-03.parquet
techcatalyst-de-2026/raw/green_taxi/green_tripdata_2025-04.parquet
techcatalyst-de-2026/raw/green_taxi/green_tripdata_2025-05.parquet
techcatalyst-de-2026/raw/green_taxi/green_tripdata_2026-01.parquet
techcatalyst-de-2026/raw/green_taxi/green_tripdata_2026-02.parquet
techcatalyst-de-2026/raw/green_taxi/green_tripdata_2026-03.parquet
techcatalyst-de-2026/raw/green_taxi/green_tripdata_2026-04.parquet
techcatalyst-de-2026/raw/green_taxi/green_tripdata_2026-05.parquet
techcatalyst-de-2026/raw/weather/weather_raw.parquet
techcatalyst-de-2026/raw/yellow_taxi/yellow_tripdata_2025-01.parquet
techcatalyst-de-2026/raw/yellow_taxi/yellow_tripdata_2025-02.parquet
techcatalyst-de-2026/raw/yellow_taxi/yellow_tripdata_2025-03.parquet
techcatalyst-de-2026/raw/yellow_taxi/yellow_tripdata_2025-04.parquet
t

#### Verify some files

In [ ]:
df = pl.read_parquet("s3://techcatalyst-de-2026/raw/green_taxi/green_tripdata_2025-01.parquet")

df.head()

VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
i32,datetime[μs],datetime[μs],str,i64,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,f64,f64
2,2025-01-01 00:03:01,2025-01-01 00:17:12,"""N""",1,75,235,1,5.93,24.7,1.0,0.5,6.8,0.0,null,1.0,34.0,1,1,0.0,0.0
2,2025-01-01 00:19:59,2025-01-01 00:25:52,"""N""",1,166,75,1,1.32,8.6,1.0,0.5,0.0,0.0,null,1.0,11.1,2,1,0.0,0.0
2,2025-01-01 00:05:29,2025-01-01 00:07:21,"""N""",5,171,73,1,0.41,25.55,0.0,0.0,0.0,0.0,null,1.0,26.55,2,2,0.0,0.0
2,2025-01-01 00:52:24,2025-01-01 01:07:52,"""N""",1,74,223,1,4.12,21.2,1.0,0.5,6.13,6.94,null,1.0,36.77,1,1,0.0,0.0
2,2025-01-01 00:25:05,2025-01-01 01:01:10,"""N""",1,66,158,1,4.71,33.8,1.0,0.5,7.81,0.0,null,1.0,46.86,1,1,2.75,0.0


In [15]:
df.shape

(48326, 21)

In [24]:
for col in df.columns:
    print(
        f"{col}: "
        f"distinct={df[col].n_unique()}, "
        f"nulls={df[col].null_count()}"
    )


VendorID: distinct=2, nulls=0
lpep_pickup_datetime: distinct=47420, nulls=0
lpep_dropoff_datetime: distinct=47507, nulls=0
store_and_fwd_flag: distinct=3, nulls=1836
RatecodeID: distinct=8, nulls=1836
PULocationID: distinct=212, nulls=0
DOLocationID: distinct=241, nulls=0
passenger_count: distinct=11, nulls=1836
trip_distance: distinct=1721, nulls=0
fare_amount: distinct=1430, nulls=0
extra: distinct=20, nulls=0
mta_tax: distinct=6, nulls=0
tip_amount: distinct=1349, nulls=0
tolls_amount: distinct=22, nulls=0
ehail_fee: distinct=1, nulls=48326
improvement_surcharge: distinct=5, nulls=0
total_amount: distinct=3940, nulls=0
payment_type: distinct=6, nulls=1836
trip_type: distinct=3, nulls=1843
congestion_surcharge: distinct=5, nulls=1836
cbd_congestion_fee: distinct=3, nulls=1836


In [40]:
df.filter(
(pl.col("congestion_surcharge") != 0)
& (pl.col("congestion_surcharge") != 2.75)
& (pl.col("congestion_surcharge") != 2.5)
).select("congestion_surcharge")

congestion_surcharge
f64
-2.75


Upon inspecting this surcharge column we found that there is potential for negative values in the surcharge or fee columns

In [ ]:
df.filter(
    (pl.col("cbd_congestion_fee") < 0)
).select("cbd_congestion_fee")

We noticed a mismatch in the dropoff and pickup counts. Below we discovered some negative values that should not be negative.

In [90]:
df.filter(
            (pl.col("passenger_count") > 1)
        ).select(["passenger_count","lpep_dropoff_datetime", "DOLocationID"]).unique()




passenger_count,lpep_dropoff_datetime,DOLocationID
i64,datetime[μs],i32
2,2025-01-16 18:24:33,75
2,2025-01-22 17:34:19,114
2,2025-01-05 13:36:38,138
2,2025-01-23 08:18:29,155
2,2025-01-06 22:00:35,255
…,…,…
2,2025-01-18 19:51:51,72
2,2025-01-20 12:52:00,142
5,2025-01-05 18:41:41,129


In [95]:
df2 = pl.read_parquet("s3://techcatalyst-de-2026/raw/yellow_taxi/yellow_tripdata_2025-01.parquet")
df1 = pl.read_parquet("s3://techcatalyst-de-2026/raw/green_taxi/green_tripdata_2025-01.parquet")

df2.head()

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
i32,datetime[μs],datetime[μs],i64,f64,i64,str,i32,i32,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1,2025-01-01 00:18:38,2025-01-01 00:26:59,1,1.6,1,"""N""",229,237,1,10.0,3.5,0.5,3.0,0.0,1.0,18.0,2.5,0.0,0.0
1,2025-01-01 00:32:40,2025-01-01 00:35:13,1,0.5,1,"""N""",236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
1,2025-01-01 00:44:04,2025-01-01 00:46:01,1,0.6,1,"""N""",141,141,1,5.1,3.5,0.5,2.0,0.0,1.0,12.1,2.5,0.0,0.0
2,2025-01-01 00:14:27,2025-01-01 00:20:01,3,0.52,1,"""N""",244,244,2,7.2,1.0,0.5,0.0,0.0,1.0,9.7,0.0,0.0,0.0
2,2025-01-01 00:21:34,2025-01-01 00:25:06,3,0.66,1,"""N""",244,116,2,5.8,1.0,0.5,0.0,0.0,1.0,8.3,0.0,0.0,0.0


Want to check if Green and Yellow have the same columns and schema.

In [97]:
schema1 = df1.schema
schema2 = df2.schema

for col in set(schema1) | set(schema2):
    if schema1.get(col) != schema2.get(col):
        print(
            f"{col}: "
            f"df1={schema1.get(col)} "
            f"df2={schema2.get(col)}"
        )

lpep_pickup_datetime: df1=Datetime(time_unit='us', time_zone=None) df2=None
lpep_dropoff_datetime: df1=Datetime(time_unit='us', time_zone=None) df2=None
Airport_fee: df1=None df2=Float64
tpep_pickup_datetime: df1=None df2=Datetime(time_unit='us', time_zone=None)
trip_type: df1=Int64 df2=None
tpep_dropoff_datetime: df1=None df2=Datetime(time_unit='us', time_zone=None)
ehail_fee: df1=Float64 df2=None


Here the schemas appear different. Lpep and Tpep is just a naming convetion. 
1. Green does not take passengers to the airport
2. Trip type is not a column in yellow
3. Yellow does not have ehail options.

Checking the counts of each file

In [103]:
files = [f"s3://{f}" for f in files]

for file in files:
    count = (
        pl.scan_parquet(file)
        .select(pl.len())
        .collect()
        .item()
    )
    print(file, count)

s3://techcatalyst-de-2026/raw/green_taxi/green_tripdata_2025-01.parquet 48326
s3://techcatalyst-de-2026/raw/green_taxi/green_tripdata_2025-02.parquet 46621
s3://techcatalyst-de-2026/raw/green_taxi/green_tripdata_2025-03.parquet 51539
s3://techcatalyst-de-2026/raw/green_taxi/green_tripdata_2025-04.parquet 52132
s3://techcatalyst-de-2026/raw/green_taxi/green_tripdata_2025-05.parquet 55399
s3://techcatalyst-de-2026/raw/green_taxi/green_tripdata_2026-01.parquet 40272
s3://techcatalyst-de-2026/raw/green_taxi/green_tripdata_2026-02.parquet 37373
s3://techcatalyst-de-2026/raw/green_taxi/green_tripdata_2026-03.parquet 44208
s3://techcatalyst-de-2026/raw/green_taxi/green_tripdata_2026-04.parquet 44238
s3://techcatalyst-de-2026/raw/green_taxi/green_tripdata_2026-05.parquet 44921
s3://techcatalyst-de-2026/raw/weather/weather_raw.parquet 20
s3://techcatalyst-de-2026/raw/yellow_taxi/yellow_tripdata_2025-01.parquet 3475226
s3://techcatalyst-de-2026/raw/yellow_taxi/yellow_tripdata_2025-02.parquet 357

In [121]:
test2 = pl.read_parquet("s3://techcatalyst-de-2026/raw/yellow_taxi/yellow_tripdata_2025-01.parquet")
test2.select(pl.col("tpep_pickup_datetime"))


tpep_pickup_datetime
datetime[μs]
2025-01-01 00:18:38
2025-01-01 00:32:40
2025-01-01 00:44:04
2025-01-01 00:14:27
2025-01-01 00:21:34
…
2025-01-31 23:01:48
2025-01-31 23:50:29
2025-01-31 23:26:59
